In [2]:
import pandas as pd

fact = pd.read_csv("data/star_schema/fact_track.csv")
spotify = pd.read_csv("data/raw/spotify_dataset.csv", dtype=str)

print("Spotify raw rows:       ", len(spotify))
print("Fact rows:              ", len(fact))
print("Diferencia:             ", len(spotify) - len(fact))

print("\n--- Nulls en FKs ---")
print(fact[["artist_key","album_key","genre_key","time_key"]].isnull().sum())

print("\n--- track_id únicos en spotify:", spotify["track_id"].nunique())
print("--- track_id únicos en fact:   ", fact["track_id"].nunique())

print("\n--- Duplicados (track_id, genre_key) en fact:")
print(fact.duplicated(subset=["track_id","genre_key"]).sum())

Spotify raw rows:        114000
Fact rows:               113549
Diferencia:              451

--- Nulls en FKs ---
artist_key    0
album_key     0
genre_key     0
time_key      0
dtype: int64

--- track_id únicos en spotify: 89741
--- track_id únicos en fact:    89740

--- Duplicados (track_id, genre_key) en fact:
0


In [3]:
import pandas as pd
from src.extract import run as extract_run
from src.transform_spotify import run as transform_spotify_run
from src.transform_grammys import run as transform_grammys_run
from src.merge_data import run as merge_run

spotify_raw, grammy_raw = extract_run()
spotify_clean = transform_spotify_run(spotify_raw)
grammy_clean = transform_grammys_run(grammy_raw)
merged = merge_run(spotify_clean, grammy_clean)

print("Spotify raw:    ", len(spotify_raw))
print("Spotify clean:  ", len(spotify_clean))
print("Merged:         ", len(merged))
print("track_id únicos en merged:", merged["track_id"].nunique())
print("track_genre únicos en merged:", merged["track_genre"].nunique())

python-dotenv could not parse statement starting at line 1



TASK a -- EXTRACTION

Spotify  ->  c:\Users\valka\Desktop\MyComputer\ETL\workshop2\data\raw\spotify_dataset.csv
  Shape  :  114,000 rows x 21 columns

-- Spotify sample (3 rows) --
Unnamed: 0               track_id                artists       album_name       track_name  popularity  duration_ms explicit  danceability  energy key  loudness mode  speechiness  acousticness  instrumentalness  liveness  valence  tempo time_signature track_genre
         0 5SuOikwiRyPMVoIQDJUgSV            Gen Hoshino           Comedy           Comedy          73       230666    False         0.676   0.461   1    -6.746    0       0.1430        0.0322          0.000001     0.358    0.715 87.917              4    acoustic
         1 4qPNDBW1i3p13qLCt0Ki3A           Ben Woodward Ghost (Acoustic) Ghost - Acoustic          55       149610    False         0.420   0.166   1   -17.235    1       0.0763        0.9240          0.000006     0.101    0.267 77.489              4    acoustic
         2 1iJBSr7s7jYXzM8

In [7]:
print("Géneros en dim_genre:", len(pd.read_csv("data/star_schema/dim_genre.csv")))
print("\ntrack_genre sample en merged:")
print(merged["track_genre"].value_counts().head(5))

# El culpable real: genre_key después del map
genre_map = dict(zip(pd.read_csv("data/star_schema/dim_genre.csv")["genre_name"], pd.read_csv("data/star_schema/dim_genre.csv")["genre_key"]))
fact_genre = (
    merged["track_genre"]
    .astype(str)
    .str.split(",")
    .str[0]
    .str.strip()
    .str.lower()
)
print("\nValores únicos de genre después del split:", fact_genre.nunique())
print("Nulls después del map:", fact_genre.map(genre_map).isna().sum())
print("\nEjemplos que NO matchean:")
no_match = fact_genre[fact_genre.map(genre_map).isna()].unique()
print(no_match[:10])

Géneros en dim_genre: 114

track_genre sample en merged:
track_genre
acoustic      1000
british       1000
country       1000
disco         1000
electronic    1000
Name: count, dtype: int64

Valores únicos de genre después del split: 114
Nulls después del map: 0

Ejemplos que NO matchean:
[]


In [8]:
dim_genre = pd.read_csv("data/star_schema/dim_genre.csv")
print(dim_genre.head(20).to_string())
print("\n¿Hay espacios en genre_name?")
print(dim_genre["genre_name"].apply(lambda x: x != x.strip()).sum())
print("\nEjemplo raw de genre_name:")
print(repr(dim_genre["genre_name"].iloc[0]))

    genre_key     genre_name
0           1       acoustic
1           2       afrobeat
2           3       alt-rock
3           4    alternative
4           5        ambient
5           6          anime
6           7    black-metal
7           8      bluegrass
8           9          blues
9          10         brazil
10         11      breakbeat
11         12        british
12         13       cantopop
13         14  chicago-house
14         15       children
15         16          chill
16         17      classical
17         18           club
18         19         comedy
19         20        country

¿Hay espacios en genre_name?
0

Ejemplo raw de genre_name:
'acoustic'


In [9]:
# Verificar si el map funciona manualmente
print("¿'acoustic' está en genre_map?", "acoustic" in genre_map)
print("genre_map['acoustic'] =", genre_map.get("acoustic"))

# Verificar el tipo de las keys del map
print("\nTipo de keys en genre_map:", type(list(genre_map.keys())[0]))
print("Tipo de valores en fact_genre:", fact_genre.dtype)

# Comparar byte a byte
sample = fact_genre.iloc[0]
print(f"\nPrimer valor de fact_genre: {repr(sample)}")
print(f"¿Está en genre_map?: {sample in genre_map}")
print(f"Bytes: {sample.encode()}")
print(f"Bytes de key en dim: {list(genre_map.keys())[0].encode()}")

¿'acoustic' está en genre_map? True
genre_map['acoustic'] = 1

Tipo de keys en genre_map: <class 'str'>
Tipo de valores en fact_genre: object

Primer valor de fact_genre: 'acoustic'
¿Está en genre_map?: True
Bytes: b'acoustic'
Bytes de key en dim: b'acoustic'


In [10]:
from src.dimensional_model import _build_dim_genre

dim_genre_mem = _build_dim_genre(merged)
print("Géneros en memoria:", len(dim_genre_mem))
print(dim_genre_mem.head(5).to_string())

genre_map_mem = dict(zip(dim_genre_mem["genre_name"], dim_genre_mem["genre_key"]))

fact_genre = (
    merged["track_genre"]
    .astype(str)
    .str.split(",")
    .str[0]
    .str.strip()
    .str.lower()
)

nulls = fact_genre.map(genre_map_mem).isna().sum()
print(f"\nNulls con dim_genre en MEMORIA: {nulls}")
print(f"Nulls con dim_genre del CSV:    107701")


Géneros en memoria: 114
   genre_key   genre_name
0          1     acoustic
1          2     afrobeat
2          3     alt-rock
3          4  alternative
4          5      ambient

Nulls con dim_genre en MEMORIA: 0
Nulls con dim_genre del CSV:    107701


In [11]:
fact = pd.read_csv("data/star_schema/fact_track.csv")
print("Fact rows:", len(fact))

Fact rows: 113549
